# Pipeline V8 — Fase 2: Enriquecimento Local (12 canais + 3 escalares)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-005, T-A3-006).

Lê cada NPZ **original** (sem sufixo `_refH`/`_refV`/`_r180`) em
`dados/profundidade_minimax_11_v7_adaptativo/`, adiciona:

| Campo novo | Shape | Dtype | Descrição |
|---|---|---|---|
| `canais` | `(N, 4, 3, 12)` | int8 | 12 canais estruturais (K=11 = paridade broadcast) |
| `nomes_canais` | `(12,)` | U32 | Nomes canônicos dos 12 canais |
| `qtd_cadeias_longas` | `(N,)` | int8 | Contagem de cadeias longas abertas (≥3 caixas) |
| `total_caixas_cadeias_longas` | `(N,)` | int8 | Soma de caixas de todas as cadeias longas |
| `tamanho_max_cadeia_longa` | `(N,)` | int8 | Tamanho da maior cadeia longa |

Sobrescrita **atômica**: `.tmp.npz` + `os.replace()`. Idempotente com `FORCAR_REGRAVAR = True`.

**Pré-requisito**: `analisador_estrutural_pontinhos.py` com `N_CANAIS = 12` (T-A2-009, T-A3-002).

**Schema de saída**: v2-a3 (ver `specs/004-melhoria-geracao-dados-cnn/contracts/npz_schema.md` §3).

In [1]:
# Imports
import os
import sys
import glob
import time
import numpy as np

ROOT = os.environ.get('ARENA_SAGAZ_BACKEND', os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from gerador_dados.jogo_pontinhos.analisador_estrutural_pontinhos import (
    NOMES_CANAIS,
    N_CANAIS,
    extrair_canais,
    extrair_stats_cadeias,
)

assert N_CANAIS == 12, f'Esperado N_CANAIS=12, obtido {N_CANAIS}. Atualizar analisador (T-A2-009).'
assert len(NOMES_CANAIS) == 12
assert NOMES_CANAIS[-1] == 'paridade_cadeia_longa_impar'

print(f'N_CANAIS        = {N_CANAIS}')
print(f'NOMES_CANAIS    = {NOMES_CANAIS}')
print(f'Backend root    = {ROOT}')

N_CANAIS        = 12
NOMES_CANAIS    = ('aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita', 'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta', 'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta', 'paridade_cadeia_longa_impar')
Backend root    = d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend


In [2]:
# Parametros

DIR_NPZ = os.path.join(ROOT, 'dados', 'profundidade_minimax_11_v7_adaptativo')
PADRAO = 'dataset_pequeno_*.npz'

# Se True, regrava mesmo NPZs ja enriquecidos com 12 canais.
# Deixe True sempre que o algoritmo de canais mudar.
FORCAR_REGRAVAR = True

print(f'DIR_NPZ         = {DIR_NPZ}')
print(f'PADRAO          = {PADRAO}')
print(f'FORCAR_REGRAVAR = {FORCAR_REGRAVAR}')

DIR_NPZ         = d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minimax_11_v7_adaptativo
PADRAO          = dataset_pequeno_*.npz
FORCAR_REGRAVAR = True


In [3]:
# Descobrir NPZs originais (excluir _refH, _refV, _r180).

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')

todos = sorted(glob.glob(os.path.join(DIR_NPZ, PADRAO)))
arquivos = [
    f for f in todos
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
]

print(f'NPZs encontrados (todos)     : {len(todos)}')
print(f'NPZs originais (sem sufixo)  : {len(arquivos)}')
print(f'NPZs sufixados (ignorados)   : {len(todos) - len(arquivos)}')

NPZs encontrados (todos)     : 152
NPZs originais (sem sufixo)  : 152
NPZs sufixados (ignorados)   : 0


In [4]:
# Funcao de enriquecimento: adiciona canais (12) + 3 campos escalares.

_NOMES_CANAIS_ARR = np.array(NOMES_CANAIS, dtype='U32')

def enriquecer_arquivo(caminho: str, *, forcar: bool = False) -> dict:
    """Le NPZ, computa 12 canais + 3 escalares, regrava atomicamente.

    Returns dict com: caminho, n_estados, tinha_v2a3, tempo_s.
    """
    t0 = time.time()
    with np.load(caminho, allow_pickle=False) as f:
        d = dict(f)

    tinha_v2a3 = (
        'canais' in d
        and d['canais'].shape[-1] == N_CANAIS
        and 'qtd_cadeias_longas' in d
    )

    if tinha_v2a3 and not forcar:
        return {'caminho': caminho, 'n_estados': int(d['estados'].shape[0]),
                'tinha_v2a3': True, 'tempo_s': 0.0, 'status': 'pulado'}

    estados = d['estados']  # (N, 9, 7)
    N = estados.shape[0]

    canais_arr  = np.zeros((N, 4, 3, N_CANAIS), dtype=np.int8)
    qtd_arr     = np.zeros(N, dtype=np.int8)
    total_arr   = np.zeros(N, dtype=np.int8)
    maximo_arr  = np.zeros(N, dtype=np.int8)

    for i in range(N):
        canais_arr[i] = extrair_canais(estados[i])
        qtd, total, maximo = extrair_stats_cadeias(estados[i])
        qtd_arr[i]    = qtd
        total_arr[i]  = total
        maximo_arr[i] = maximo

    novo = dict(d)
    novo['canais']                       = canais_arr
    novo['nomes_canais']                 = _NOMES_CANAIS_ARR
    novo['qtd_cadeias_longas']           = qtd_arr
    novo['total_caixas_cadeias_longas']  = total_arr
    novo['tamanho_max_cadeia_longa']     = maximo_arr

    tmp = caminho + '.tmp.npz'
    np.savez_compressed(tmp, **novo)
    os.replace(tmp, caminho)

    return {'caminho': caminho, 'n_estados': N, 'tinha_v2a3': tinha_v2a3,
            'tempo_s': time.time() - t0, 'status': 'enriquecido'}

print('Funcao enriquecer_arquivo() pronta.')

Funcao enriquecer_arquivo() pronta.


In [5]:
# Loop principal.

relatorio = []
for arq in arquivos:
    info = enriquecer_arquivo(arq, forcar=FORCAR_REGRAVAR)
    flag = '*' if info['status'] == 'enriquecido' else ' '
    print(
        f"{flag} {os.path.basename(info['caminho'])}: "
        f"N={info['n_estados']} "
        f"status={info['status']} "
        f"t={info['tempo_s']:.2f}s"
    )
    relatorio.append(info)

n_enriquecidos = sum(1 for r in relatorio if r['status'] == 'enriquecido')
n_pulados      = sum(1 for r in relatorio if r['status'] == 'pulado')
tempo_total    = sum(r['tempo_s'] for r in relatorio)

print()
print(f'Total processados : {len(relatorio)}')
print(f'  Enriquecidos    : {n_enriquecidos}')
print(f'  Pulados (ja ok) : {n_pulados}')
print(f'Tempo total       : {tempo_total:.1f}s')

* dataset_pequeno_0001.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0002.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0003.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0004.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0005.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0006.npz: N=5000 status=enriquecido t=0.46s
* dataset_pequeno_0007.npz: N=5000 status=enriquecido t=0.45s
* dataset_pequeno_0008.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0009.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0010.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0011.npz: N=5000 status=enriquecido t=0.45s
* dataset_pequeno_0012.npz: N=5000 status=enriquecido t=0.45s
* dataset_pequeno_0013.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0014.npz: N=5000 status=enriquecido t=0.44s
* dataset_pequeno_0015.npz: N=5000 status=enriquecido t=0.45s
* dataset_pequeno_0016.npz: N=5000 status=enriquecido t=0.44s
* datase

In [6]:
# Auditoria pos-enriquecimento.
# Gate T-A3-006: todos os 152 NPZs com canais.shape==(N,4,3,12) + 3 escalares.

erros = []
for arq in arquivos:
    with np.load(arq, allow_pickle=False) as d:
        # canais
        if 'canais' not in d.files:
            erros.append(f'{os.path.basename(arq)}: campo canais ausente')
            continue
        shp = d['canais'].shape
        if shp[1:] != (4, 3, 12):
            erros.append(f'{os.path.basename(arq)}: canais.shape={shp} (esperado (N,4,3,12))')
        if d['canais'].dtype != np.int8:
            erros.append(f'{os.path.basename(arq)}: canais.dtype={d["canais"].dtype} (esperado int8)')
        # nomes_canais
        if not np.array_equal(d['nomes_canais'], _NOMES_CANAIS_ARR):
            erros.append(f'{os.path.basename(arq)}: nomes_canais divergente')
        # escalares
        for campo in ('qtd_cadeias_longas', 'total_caixas_cadeias_longas', 'tamanho_max_cadeia_longa'):
            if campo not in d.files:
                erros.append(f'{os.path.basename(arq)}: campo {campo} ausente')
        # sem tmp residual
        if os.path.exists(arq + '.tmp.npz'):
            erros.append(f'{os.path.basename(arq)}: .tmp.npz residual encontrado')

if erros:
    print(f'FALHA: {len(erros)} erro(s):')
    for e in erros:
        print(f'  {e}')
else:
    total_amostras = sum(
        np.load(a, allow_pickle=False)['estados'].shape[0] for a in arquivos
    )
    print(f'OK: {len(arquivos)} NPZs auditados — {total_amostras:,} estados totais.')
    print(f'Schema v2-a3 validado: canais(N,4,3,12) + 3 escalares em todos os arquivos.')

OK: 152 NPZs auditados — 758,640 estados totais.
Schema v2-a3 validado: canais(N,4,3,12) + 3 escalares em todos os arquivos.
